In [1]:
# ============================================
# STEP 1: Setup — Imports, Paths, Configuration
# ============================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

RANDOM_STATE = 42

PROJECT_ROOT = Path("../")
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Report: T2-B - Plausible Identifier Resampling Robustness")
print("Models: Decision Tree, LightGBM (Set A and Set B)")
print("Random state:", RANDOM_STATE)

Report: T2-B - Plausible Identifier Resampling Robustness
Models: Decision Tree, LightGBM (Set A and Set B)
Random state: 42


## T2-B: Plausible Identifier Resampling Robustness

### Hypothesis H3

Set A models will still show a meaningful Macro-F1 drop under T2-B (plausible resampling) compared to the clean T1 result, but a **smaller drop than T2-A** (pure permutation) — because T2-B draws replacement identifier values from the same empirical distribution each column already has (same pool of real, observed values), rather than fully scrambling the row-level pairing in a way that could coincidentally still preserve some structure. Both tests should show degradation; the comparison between them indicates whether the model's vulnerability is specific to naive full permutation or extends to more realistic attacker identifier manipulation.

### T2-A vs T2-B — what's actually different

Both T2-A and T2-B break the row-wise pairing between identifiers and their true attack label, using the same four columns (`src_ip`, `src_port`, `dst_ip`, `dst_port`). The difference is how the replacement values are chosen:

- **T2-A (permutation):** each column's existing values are shuffled across rows (sampling without replacement from that column's own values) — every original value is reused exactly once, just in a different row.
- **T2-B (plausible resampling):** each column's replacement values are drawn **with replacement** from that column's empirical distribution (`.sample(..., replace=True)`) — so some values can repeat or be skipped, more closely simulating an attacker who draws a spoofed identifier from a plausible pool (e.g. a common subnet range) rather than a clean one-to-one reshuffle of the test set's own rows.

### Protocol

- Same locked T1 test set, same saved Decision Tree / LightGBM (Set A) models, no retraining
- `src_ip`, `src_port`, `dst_ip`, `dst_port` independently resampled with replacement from their own empirical (test-set) distribution
- Compared against both T1 (clean) and T2-A (permutation) results

In [2]:
# ============================================
# STEP 2: Recreate Test Set + Apply Unified 5-Identifier Resampling
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

FEATURE_SETS_PATH = PROJECT_ROOT / "data" / "processed" / "feature_sets.json"
with open(FEATURE_SETS_PATH, "r") as f:
    feature_sets = json.load(f)

all_features = feature_sets["set_a_columns"]
set_b_features = feature_sets["set_b_columns"]
identifier_cols = feature_sets["excluded_identifiers"]

SET_A_PATH = PROJECT_ROOT / "data" / "processed" / "set_a_identifier_inclusive.csv"
df = pd.read_csv(SET_A_PATH)

X = df[all_features].copy()
y = df["type"].copy()

# Split first
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
for col in categorical_cols:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_val[col] = X_val[col].fillna("missing").astype(str)
    X_test[col] = X_test[col].fillna("missing").astype(str)

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
X_val[categorical_cols] = encoder.transform(X_val[categorical_cols])
X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])

X_test_raw = X_test.copy()
X_test_filled = X_test.fillna(-1)

# Apply Joint Identifier Tuple Resampling (All 5 Identifiers Synchronized)
rng = np.random.default_rng(RANDOM_STATE)
n_rows = len(X_test_raw)

sampled_positions = rng.choice(np.arange(n_rows), size=n_rows, replace=True)
sampled_tuples = X_test_raw.iloc[sampled_positions][identifier_cols].to_numpy()

X_test_raw_resamp = X_test_raw.copy()
X_test_filled_resamp = X_test_filled.copy()
X_test_raw_resamp.loc[:, identifier_cols] = sampled_tuples
X_test_filled_resamp.loc[:, identifier_cols] = sampled_tuples

print("Resampling method: Joint identifier tuple resampling across all 5 identifiers")
assert X_test_raw_resamp[set_b_features].equals(X_test_raw[set_b_features])
print("Sanity checks passed: Set B behavioural features remain 100% identical.")


Resampling method: Joint identifier tuple resampling across all 5 identifiers
Sanity checks passed: Set B behavioural features remain 100% identical.


C:\Users\User\AppData\Local\Temp\ipykernel_22600\1869593757.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns.tolist()


In [3]:
# ============================================
# STEP 4: Evaluate T2-B Tuple Resampling (Set A and Set B)
# ============================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_files_t2b = {
    ("Decision Tree", "Set A"): ("dt_set_a.joblib", X_test_raw, X_test_raw_resamp, all_features),
    ("Decision Tree", "Set B"): ("dt_set_b.joblib", X_test_raw, X_test_raw_resamp, set_b_features),
    ("LightGBM", "Set A"): ("lgbm_set_a.joblib", X_test_filled, X_test_filled_resamp, all_features),
    ("LightGBM", "Set B"): ("lgbm_set_b.joblib", X_test_filled, X_test_filled_resamp, set_b_features),
}

t2b_results = []

for (model_name, feat_set), (filename, clean_set, resamp_set, features) in model_files_t2b.items():
    model = joblib.load(MODELS_DIR / filename)
    clean_preds = model.predict(clean_set[features])
    resamp_preds = model.predict(resamp_set[features])

    clean_f1 = f1_score(y_test, clean_preds, average="macro", zero_division=0)
    resamp_f1 = f1_score(y_test, resamp_preds, average="macro", zero_division=0)

    if feat_set == "Set B":
        assert np.array_equal(clean_preds, resamp_preds), "Set B predictions altered!"
        print(f"[Verified] {model_name} Set B prediction equality under T2-B: 100%")

    t2b_results.append({
        "Model": model_name,
        "Feature Set": feat_set,
        "T1 Clean Macro-F1": clean_f1,
        "T2-B Tuple Macro-F1": resamp_f1,
        "Drop vs T1": round(clean_f1 - resamp_f1, 4),
        "Accuracy": accuracy_score(y_test, resamp_preds),
        "Precision": precision_score(y_test, resamp_preds, average="macro", zero_division=0),
        "Recall": recall_score(y_test, resamp_preds, average="macro", zero_division=0),
    })

t2b_df = pd.DataFrame(t2b_results)
print("\n=== T2-B Joint Tuple Resampling Results ===")
print(t2b_df.round(4).to_string(index=False))

t2b_df.to_csv(RESULTS_DIR / "t2b_tuple_resampling_results.csv", index=False)
print("\nSaved to:", RESULTS_DIR / "t2b_tuple_resampling_results.csv")


[Verified] Decision Tree Set B prediction equality under T2-B: 100%
[Verified] LightGBM Set B prediction equality under T2-B: 100%

=== T2-B Joint Tuple Resampling Results ===
        Model Feature Set  T1 Clean Macro-F1  T2-B Tuple Macro-F1  Drop vs T1  Accuracy  Precision  Recall
Decision Tree       Set A             0.9806               0.2386      0.7420    0.2685     0.2681  0.2352
Decision Tree       Set B             0.9471               0.9471      0.0000    0.9761     0.9472  0.9472
     LightGBM       Set A             0.9861               0.4732      0.5129    0.5089     0.5030  0.4662
     LightGBM       Set B             0.8756               0.8756      0.0000    0.9237     0.8758  0.8912

Saved to: ..\results\t2b_tuple_resampling_results.csv


## T2-B Result & H3 Conclusion

### Side-by-Side Comparison of Robustness Regimes (Seed 42)
| Model | Feature Set | T1 Clean | T2-A (Permutation) | T2-B (Tuple Resampling) | Outcome |
|---|:---:|:---:|:---:|:---:|:---:|
| Decision Tree | Set A | 0.9806 | 0.2252 | 0.2386 | Severe Collapse |
| **Decision Tree** | **Set B** | **0.9471** | **0.9471** | **0.9471** | **Invariant (Δ = 0.0)** |
| LightGBM | Set A | 0.9861 | 0.5380 | **0.4732** | **Worse than Permutation (-6.5 pp)** |
| **LightGBM** | **Set B** | **0.8756** | **0.8756** | **0.8756** | **Invariant (Δ = 0.0)** |

**Hypothesis H3: Rejected (original smaller-drop prediction refuted; deceptive-shortcut effect confirmed).**
The hypothesis that plausible, distribution-matched resampling would soften the Macro-F1 drop compared to naive permutation is rejected — for LightGBM, joint-tuple resampling causes a slightly *larger* drop (0.5129) than pure permutation (0.4481). Injecting coherent identifier tuples (rather than independently-shuffled values) appears to act as an active deceptive shortcut, triggering memorized decision paths that confidently misclassify flows into the donor tuple's label. Set B remains fully invariant under both perturbation types, reinforcing the case for identifier exclusion. Decision Tree shows a smaller, comparable drop under both regimes (0.7553 vs 0.7420).

##REMOVE BELOW

## T2-B Result & H3 Conclusion

### Side-by-Side Comparison of Robustness Regimes (Seed 42)
| Model | Feature Set | T1 Clean | T2-A (Permutation) | T2-B (Tuple Resampling) | Outcome |
|---|:---:|:---:|:---:|:---:|:---:|
| Decision Tree | Set A | 0.9810 | 0.2237 | 0.2387 | Severe Collapse |
| **Decision Tree** | **Set B** | **0.9471** | **0.9471** | **0.9471** | **Invariant (Δ = 0.0)** |
| LightGBM | Set A | 0.9861 | 0.5476 | **0.4785** | **Worse than Permutation (-6.9 pp)** |
| **LightGBM** | **Set B** | **0.8756** | **0.8756** | **0.8756** | **Invariant (Δ = 0.0)** |

### Multi-Seed Findings (5 Seeds: Mean ± Std)
- **Decision Tree Set A:** T2-A ($0.2446 \pm 0.0109$) $\approx$ T2-B ($0.2534 \pm 0.0073$).
- **LightGBM Set A:** T2-A ($0.5624 \pm 0.0156$) $\rightarrow$ T2-B drops further to **$0.5009 \pm 0.0144$** (Drop: $0.4863$).

**Hypothesis H3: Rejected (Original smaller-drop prediction refuted; Deceptive Shortcut confirmed).**
The naive hypothesis that plausible resampling would soften performance degradation is rejected. Under independent resampling, low cardinality causes with-replacement draws to converge to permutation. Under joint-tuple resampling, injecting coherent identifier tuples acts as an **active deceptive shortcut**, triggering deep memorized decision paths that confidently misclassify flows into the donor tuple's label. This confirms that identifier reliance cannot be salvaged by assuming plausible attacker distributions, providing decisive justification for the complete exclusion of identifiers (Set B).